In [1]:
import glob
import os
import numpy as np
import pandas as pd

folder = r"C:\Users\ASUS\Desktop\volPredict"
end_date = pd.Timestamp("2025-06-30")


def load_cleaned_options(pattern):
    files = glob.glob(os.path.join(folder, pattern))
    if not files:
        raise FileNotFoundError(
            f"No files found for pattern '{pattern}' in {folder}"
        )

    df_list = []
    for f in files:
        temp = pd.read_csv(f, low_memory=False)
        temp.columns = temp.columns.str.strip()
        df_list.append(temp)

    df = pd.concat(df_list, ignore_index=True)

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df["Expiry"] = pd.to_datetime(df["Expiry"], dayfirst=True, errors="coerce")
    df = df[df["Date"] <= end_date].copy()

    numeric_cols = [
        "Strike Price",
        "Close",
        "No. of contracts",
        "Open Int",
        "Underlying Value",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["DTE"] = (df["Expiry"] - df["Date"]).dt.days
    return df


print("=== LOADING DATA & ANALYZING EXPIRY SCHEDULE ===")
ce = load_cleaned_options("OPTIDX_NIFTY_CE_*.csv")
pe = load_cleaned_options("OPTIDX_NIFTY_PE_*.csv")

print(f"✓ Loaded CE: {len(ce):,} rows | PE: {len(pe):,} rows\n")

# 1. Unique Expiry Day-of-Week Distribution
ce_expiries = ce[["Expiry"]].drop_duplicates().copy()
ce_expiries["Weekday"] = ce_expiries["Expiry"].dt.day_name()

print("--- Unique Expiry Day-of-Week Distribution ---")
print(ce_expiries["Weekday"].value_counts())
print()

# 2. Expiries Per Trading Day Distribution
exp_per_day = (
    ce.groupby("Date")["Expiry"].nunique().reset_index(name="Available_Expiries")
)

print("--- Available Expiries Per Trading Day (Distribution) ---")
print(exp_per_day["Available_Expiries"].value_counts().sort_index())

=== LOADING DATA & ANALYZING EXPIRY SCHEDULE ===
✓ Loaded CE: 1,040,344 rows | PE: 1,075,736 rows

--- Unique Expiry Day-of-Week Distribution ---
Weekday
Thursday     240
Wednesday     13
Tuesday        1
Name: count, dtype: int64

--- Available Expiries Per Trading Day (Distribution) ---
Available_Expiries
18    557
19     67
20      4
21    454
22     31
Name: count, dtype: int64


In [2]:
print("=== APPLYING LIQUIDITY & MONEYNESS FILTERS ===")

ce["Moneyness"] = ce["Strike Price"] / ce["Underlying Value"]
pe["Moneyness"] = pe["Strike Price"] / pe["Underlying Value"]

# Defined liquidity and validity thresholds
VOLUME_MIN = 10  # Minimum contracts traded
OI_MIN = 100  # Minimum open interest
PRICE_MIN = 0.50  # Minimum option premium (in INR)
MONEYNESS_MIN = 0.85  # 85% of spot
MONEYNESS_MAX = 1.15  # 115% of spot
DTE_MIN = 2  # Exclude DTE < 2 days to prevent gamma/vega explosion

ce_liquid = ce[
    (ce["No. of contracts"] >= VOLUME_MIN)
    & (ce["Open Int"] >= OI_MIN)
    & (ce["Close"] >= PRICE_MIN)
    & (ce["Moneyness"] >= MONEYNESS_MIN)
    & (ce["Moneyness"] <= MONEYNESS_MAX)
    & (ce["DTE"] >= DTE_MIN)
].copy()

pe_liquid = pe[
    (pe["No. of contracts"] >= VOLUME_MIN)
    & (pe["Open Int"] >= OI_MIN)
    & (pe["Close"] >= PRICE_MIN)
    & (pe["Moneyness"] >= MONEYNESS_MIN)
    & (pe["Moneyness"] <= MONEYNESS_MAX)
    & (pe["DTE"] >= DTE_MIN)
].copy()

print(
    f"Calls (CE) : {len(ce):,} raw rows -> {len(ce_liquid):,} liquid rows ({len(ce_liquid)/len(ce):.1%})"
)
print(
    f"Puts (PE)  : {len(pe):,} raw rows -> {len(pe_liquid):,} liquid rows ({len(pe_liquid)/len(pe):.1%})"
)

=== APPLYING LIQUIDITY & MONEYNESS FILTERS ===
Calls (CE) : 1,040,344 raw rows -> 190,302 liquid rows (18.3%)
Puts (PE)  : 1,075,736 raw rows -> 196,640 liquid rows (18.3%)


In [3]:
print("=== INSPECTING A SAMPLE TRADING DAY (2025-06-18) ===")

test_date = pd.Timestamp("2025-06-18")

sample_ce = (
    ce_liquid[ce_liquid["Date"] == test_date][
        [
            "Expiry",
            "DTE",
            "Strike Price",
            "Underlying Value",
            "Moneyness",
            "Close",
            "No. of contracts",
        ]
    ]
    .groupby(["Expiry", "DTE"])
    .agg(
        Strikes_Count=("Strike Price", "count"),
        Min_Moneyness=("Moneyness", "min"),
        Max_Moneyness=("Moneyness", "max"),
        Total_Volume=("No. of contracts", "sum"),
    )
    .reset_index()
    .sort_values("DTE")
)

print(f"Trading Date: {test_date.strftime('%Y-%m-%d')}\n")
print(sample_ce.to_string(index=False))

=== INSPECTING A SAMPLE TRADING DAY (2025-06-18) ===
Trading Date: 2025-06-18

    Expiry  DTE  Strikes_Count  Min_Moneyness  Max_Moneyness  Total_Volume
2025-06-26    8             77       0.866514       1.128484     2283477.0
2025-07-03   15             53       0.967272       1.084151      136112.0
2025-07-10   22             37       0.963242       1.084151       11923.0
2025-07-17   29             22       0.987423       1.053923        1756.0
2025-07-31   43             57       0.896742       1.084151      150265.0
2025-08-28   71             28       0.926969       1.072060       10153.0
2025-09-25   99              7       0.886666       1.128484        7569.0
2025-12-24  189              7       0.886666       1.128484        7793.0


In [4]:
print("=== SAVING LIQUID DATASETS ===")

ce_liquid["Type"] = "CE"
pe_liquid["Type"] = "PE"

# Combine both into a single unified dataframe
options_liquid = pd.concat([ce_liquid, pe_liquid], ignore_index=True)

# Keep only necessary columns for IV calculation
cols_to_keep = [
    "Date", "Expiry", "DTE", "Type", "Strike Price", 
    "Underlying Value", "Close", "No. of contracts", "Moneyness"
]
options_liquid = options_liquid[cols_to_keep].copy()

# Save to CSV (or Parquet if you prefer faster loading)
save_path = os.path.join(folder, "options_liquid.csv")
options_liquid.to_csv(save_path, index=False)

print(f"✓ Saved {len(options_liquid):,} combined liquid rows to: {save_path}")

=== SAVING LIQUID DATASETS ===
✓ Saved 386,942 combined liquid rows to: C:\Users\ASUS\Desktop\volPredict\options_liquid.csv
